# 🌿 Mint Leaf AI — STEP 2: Dataset Audit Module

Welcome to **Step 2** of the Mint Leaf AI project. This notebook (`01_dataset_audit.ipynb`) implements an automated, non-destructive audit of all raw image sources in `data/raw/`.

### 🎯 Objectives & Key Deliverables:
1. **Automatic Folder & Class Detection**: Recursively scan all raw sources (`Mint leaf`, `Mentha (Mint)`, `Fresh`, `Spoiled`, `Dried`, `Augmented Mint Leaf`).
2. **Image Integrity Audit**: Test every image for corruption or unreadable data (via PIL & OpenCV).
3. **Metadata & Resolution Analysis**: Extract Width, Height, Aspect Ratio, Color Mode, and identify the Most Common Image Resolution.
4. **Exact Duplicate Detection**: Use MD5 byte hashing to detect exact duplicate files across folders without modifying/deleting any source file.
5. **Visual Dashboards**: Generate class distribution charts and a sample image inspection grid.
6. **Master Image Inventory & JSON Report**: Export `outputs/reports/master_image_inventory.csv` and `outputs/reports/dataset_audit_report.json`.

--- 
⚠️ **Constraint Checklist**:  
- [x] No original images modified, renamed, moved, or deleted.  
- [x] No train/validation/test splitting.  
- [x] No dataset augmentation.  
- [x] No model training.  
- [x] Raw folders treated purely as raw data sources (no premature label assumptions).

## 🛠️ Section 1: Environment & Directory Resolution

In [ ]:
import os
import sys
import glob
import json
import time
import hashlib
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from tqdm import tqdm

# Set visualization aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.autolayout"] = True

# Environment Detection
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running in Google Colab Environment.")
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/mint-leaf-ai')
else:
    print("💻 Running in Local Antigravity IDE Environment.")
    cwd = Path(os.getcwd()).resolve()
    BASE_PATH = cwd.parent if cwd.name == 'notebooks' else cwd

DATA_RAW_DIR = BASE_PATH / 'data' / 'raw'
OUTPUT_REPORTS_DIR = BASE_PATH / 'outputs' / 'reports'
OUTPUT_VIS_DIR = BASE_PATH / 'outputs' / 'visualizations'

OUTPUT_REPORTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIS_DIR.mkdir(parents=True, exist_ok=True)

# Smart Root Path Resolution (handles data/raw/ or data/raw/main mint lead dataset/)
TARGET_FOLDERS = {"Mint leaf", "Mentha (Mint)", "Fresh", "Spoiled", "Dried", "Augmented Mint Leaf"}
DATASET_ROOT = None

for root, dirs, files in os.walk(DATA_RAW_DIR):
    if TARGET_FOLDERS.intersection(set(dirs)):
        DATASET_ROOT = Path(root)
        break

if DATASET_ROOT is None:
    DATASET_ROOT = DATA_RAW_DIR

print(f"📂 Workspace Path:     {BASE_PATH}")
print(f"🖼️ Raw Dataset Root:   {DATASET_ROOT}")
print(f"✓ Dataset Path Exists: {DATASET_ROOT.exists()}")

## 🔍 Section 2: Recursive File Scanner & Metadata Extraction

In [ ]:
SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

inventory_records = []
folder_image_counts = defaultdict(int)
format_distribution = defaultdict(int)
resolution_list = []
hash_map = defaultdict(list)
corrupted_image_records = []

print(f"🔍 Recursively scanning image files under {DATASET_ROOT}...")

# Non-destructive scan
for root, dirs, files in os.walk(DATASET_ROOT):
    rel_dir = Path(root).relative_to(DATASET_ROOT)
    parts = rel_dir.parts
    if not parts:
        continue
        
    top_folder_class = parts[0]
    sub_path_str = " / ".join(parts[1:]) if len(parts) > 1 else "Root"
    
    for filename in files:
        ext = Path(filename).suffix.lower()
        if ext in SUPPORTED_EXTENSIONS:
            abs_file_path = Path(root) / filename
            rel_file_path = abs_file_path.relative_to(BASE_PATH)
            
            # 1. MD5 File Hash Calculation (Non-destructive)
            file_hash = None
            try:
                with open(abs_file_path, 'rb') as f:
                    file_bytes = f.read()
                    file_hash = hashlib.md5(file_bytes).hexdigest()
            except Exception as e:
                corrupted_image_records.append({
                    'filepath': str(rel_file_path),
                    'class_name': top_folder_class,
                    'error': f"File Read Error: {e}"
                })
                continue
            
            # 2. Image Reading & Integrity Audit
            width, height, aspect_ratio, color_mode = None, None, None, None
            is_valid = True
            
            # PIL Verification
            try:
                with Image.open(abs_file_path) as img:
                    img.verify()
                with Image.open(abs_file_path) as img:
                    width, height = img.size
                    color_mode = img.mode
                    aspect_ratio = round(width / height, 3)
                    resolution_list.append((width, height))
            except Exception as e:
                is_valid = False
                corrupted_image_records.append({
                    'filepath': str(rel_file_path),
                    'class_name': top_folder_class,
                    'error': f"PIL Corrupt Image Error: {e}"
                })
                continue
            
            # OpenCV Verification
            if is_valid:
                mat = cv2.imread(str(abs_file_path))
                if mat is None:
                    corrupted_image_records.append({
                        'filepath': str(rel_file_path),
                        'class_name': top_folder_class,
                        'error': "OpenCV Read Error: Returned None matrix"
                    })
                    continue
            
            # Aggregate Statistics
            folder_image_counts[top_folder_class] += 1
            format_distribution[ext] += 1
            hash_map[file_hash].append(str(rel_file_path))
            
            inventory_records.append({
                'class_name': top_folder_class,
                'filename': filename,
                'path': str(rel_file_path),
                'file_extension': ext,
                'width': width,
                'height': height,
                'aspect_ratio': aspect_ratio,
                'color_mode': color_mode,
                'image_hash': file_hash
            })

df_master = pd.DataFrame(inventory_records)
total_images = len(df_master)

print(f"\n✅ Inventory Extraction Complete!")
print(f"- Total Valid Images Processed: {total_images:,}")
print(f"- Total Corrupted Files:       {len(corrupted_image_records):,}")

## 📊 Section 3: Class Distribution, Resolutions & Formats

In [ ]:
# Folder Image Breakdown
folder_summary_rows = []
for folder in ["Mint leaf", "Mentha (Mint)", "Fresh", "Spoiled", "Dried", "Augmented Mint Leaf"]:
    cnt = folder_image_counts.get(folder, 0)
    pct = (cnt / total_images * 100) if total_images > 0 else 0.0
    folder_summary_rows.append({
        'Folder / Source Class': folder,
        'Image Count': cnt,
        'Percentage (%)': round(pct, 2)
    })

df_folder_summary = pd.DataFrame(folder_summary_rows)
print("📊 Raw Folder Image Counts:")
display(df_folder_summary)

# Format Distribution
df_formats = pd.DataFrame(list(format_distribution.items()), columns=['Extension', 'Count'])
df_formats['Percentage (%)'] = (df_formats['Count'] / total_images * 100).round(2)
print("\n📸 Image Extension Distribution:")
display(df_formats)

# Most Common Resolution
most_common_tuple = Counter(resolution_list).most_common(1)
if most_common_tuple:
    mc_w, mc_h = most_common_tuple[0][0]
    mc_count = most_common_tuple[0][1]
    most_common_res_str = f"{mc_w} × {mc_h}"
    print(f"\n📏 Most Common Resolution: {most_common_res_str} ({mc_count:,} occurrences)")
else:
    most_common_res_str = "N/A"

## 👯 Section 4: Duplicate Image Detection (Non-Destructive)

In [ ]:
# Exact Duplicate Detection by Hash
duplicate_groups = {h: paths for h, paths in hash_map.items() if len(paths) > 1}
total_duplicate_files = sum(len(paths) - 1 for paths in duplicate_groups.values())

print(f"🔑 MD5 Checksum Duplicate Detection Summary:")
print(f"- Total Unique Hashes:    {len(hash_map):,}")
print(f"- Duplicate Hash Groups:  {len(duplicate_groups):,}")
print(f"- Total Exact Duplicates: {total_duplicate_files:,}")

if total_duplicate_files > 0:
    print("\n🔍 Sample Duplicate Groups (First 3 groups):")
    for i, (h, paths) in enumerate(list(duplicate_groups.items())[:3], 1):
        print(f"  Group {i} (Hash: {h[:8]}...): {len(paths)} identical copies")
        for p in paths:
            print(f"    ➔ {p}")

## 🎨 Section 5: Visual Dashboards & Representative Samples

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("🌿 Mint Leaf AI — Dataset Audit Visual Summary", fontsize=16, fontweight='bold')

# 1. Class Distribution Bar Chart
sns.barplot(data=df_folder_summary, x='Image Count', y='Folder / Source Class', ax=axes[0], palette="viridis")
axes[0].set_title("Raw Folder Image Counts", fontsize=13, fontweight='bold')
for p in axes[0].patches:
    w = p.get_width()
    axes[0].annotate(f'{int(w):,}', (w + 10, p.get_y() + p.get_height() / 2.), va='center')

# 2. Image Extension Breakdown Pie Chart
if len(df_formats) > 0:
    axes[1].pie(df_formats['Count'], labels=df_formats['Extension'], autopct='%1.1f%%', colors=sns.color_palette("mako", len(df_formats)))
    axes[1].set_title("Image Extension Distribution", fontsize=13, fontweight='bold')

plt.tight_layout()
chart_save_path = OUTPUT_VIS_DIR / 'class_distribution_dashboard.png'
plt.savefig(chart_save_path, dpi=300, bbox_inches='tight')
print(f"📊 Class distribution visualization saved to: {chart_save_path}")
plt.show()

# Representative Sample Display from Every Folder
detected_folders = [f for f in ["Mint leaf", "Mentha (Mint)", "Fresh", "Spoiled", "Dried", "Augmented Mint Leaf"] if folder_image_counts.get(f, 0) > 0]
num_folders = len(detected_folders)

if num_folders > 0:
    fig, axes = plt.subplots(2, math.ceil(num_folders / 2), figsize=(16, 8))
    fig.suptitle("🖼️ Representative Sample Images per Detected Folder", fontsize=16, fontweight='bold')
    axes_flat = axes.flatten()

    for i, folder in enumerate(detected_folders):
        sample_row = df_master[df_master['class_name'] == folder].sample(n=1, random_state=42).iloc[0]
        full_img_path = BASE_PATH / sample_row['path']
        img = Image.open(full_img_path)
        
        axes_flat[i].imshow(img)
        axes_flat[i].set_title(f"{folder}\n{sample_row['width']}x{sample_row['height']}px | {sample_row['color_mode']}", fontsize=11, fontweight='bold')
        axes_flat[i].axis('off')

    # Hide extra subplots
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].axis('off')

    plt.tight_layout()
    sample_save_path = OUTPUT_VIS_DIR / 'representative_samples_grid.png'
    plt.savefig(sample_save_path, dpi=300, bbox_inches='tight')
    print(f"🖼️ Sample images grid saved to: {sample_save_path}")
    plt.show()

## 💾 Section 6: Report Generation & Final Summary

In [ ]:
# 1. Save Master Image Inventory CSV
master_csv_path = OUTPUT_REPORTS_DIR / 'master_image_inventory.csv'
df_master.to_csv(master_csv_path, index=False)
print(f"💾 Saved Master Image Inventory CSV: {master_csv_path}")

# 2. Save Concise JSON Audit Report
json_report_data = {
    'audit_timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'total_images': total_images,
    'folder_counts': dict(folder_image_counts),
    'corrupted_images_count': len(corrupted_image_records),
    'exact_duplicates_count': total_duplicate_files,
    'most_common_resolution': most_common_res_str,
    'format_distribution': dict(format_distribution)
}

json_report_path = OUTPUT_REPORTS_DIR / 'dataset_audit_report.json'
with open(json_report_path, 'w') as jf:
    json.dump(json_report_data, jf, indent=4)
print(f"📋 Saved Audit JSON Summary:      {json_report_path}")

# Exact Final Summary Printout
print("\n" + "="*50)
print("DATASET AUDIT COMPLETE\n")
print(f"Total Images: {total_images:,}\n")
print("Folders:")
for f in ["Mint leaf", "Mentha (Mint)", "Fresh", "Spoiled", "Dried", "Augmented Mint Leaf"]:
    cnt = folder_image_counts.get(f, 0)
    prefix = "└── " if f == "Augmented Mint Leaf" else "├── "
    print(f"{prefix}{f}: {cnt:,}")

print(f"\nCorrupted: {len(corrupted_image_records)}")
print(f"Exact Duplicates: {total_duplicate_files:,}\n")
print(f"Most Common Resolution:\n{most_common_res_str}\n")
print(f"Reports:\noutputs/reports/")
print("="*50)